# Ultralytics YOLOv8 Training Notebook

## This Section is for Color Transfer

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # go one level up

from utils import normalizer, reset, validation

In [ ]:
reset.delete_all_jpg_files("/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/BulkNormalizedAnnotatedData")

In [ ]:
rfc: str = "data/IMG00425.JPG"
mtd: normalizer.TransferMethod="mean_std"

In [ ]:
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images","data/BulkNormalizedAnnotatedData/images",transfer_method=mtd)
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images/train","data/BulkNormalizedAnnotatedData/images/train",transfer_method=mtd)
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images/val","data/BulkNormalizedAnnotatedData/images/val",transfer_method=mtd)

## This Section is for Training Yolo

In [ ]:
from ultralytics import YOLO
import optuna
import time
import os

#### insert you data in the data section below

In [ ]:
from ultralytics.data.utils import check_det_dataset


dataset= "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/BulkNormalizedAnnotatedData/data.yaml"
check_det_dataset(dataset)

In [ ]:
# import yaml

# def objective(trial):
#     # Suggest hyperparameters
#     hyp = {
#         "lr0": trial.suggest_float('lr0', 1e-5, 1e-1, log=True),
#         "momentum": trial.suggest_float('momentum', 0.80, 0.99),
#         "weight_decay": trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
#         "box": trial.suggest_float('box', 0.02, 0.4),
#         "cls": trial.suggest_float('cls', 0.2, 1.0),
#         "hsv_h": trial.suggest_float('hsv_h', 0.0, 0.1),
#         "hsv_s": trial.suggest_float('hsv_s', 0.0, 0.7),
#         "hsv_v": trial.suggest_float('hsv_v', 0.0, 0.4),
#     }

#     # Save hyp file
#     hyp_path = f"trial_{trial.number}_hyp.yaml"
#     with open(hyp_path, 'w') as f:
#         yaml.dump(hyp, f)

#     try:
#         model = YOLO("yolo11n.pt")
#         results = model.train(
#             data="/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/BulkNormalizedAnnotatedData/data.yaml",
#             epochs=100,
#             imgsz=640,
#             batch=16,
#             name=f"trial_{trial.number}",
#             cfg=hyp_path  # ✅ override training config with custom hyp
#         )

#         metrics = getattr(model, "metrics", None)
#         if metrics and isinstance(metrics, dict):
#             return metrics.get("metrics/mAP50", 0.0)
#         else:
#             return 0.0

#     except Exception as e:
#         print(f"⚠️ Trial {trial.number} failed: {e}")
#         return 0.0

In [ ]:
import yaml
import optuna
from ultralytics import YOLO
import torch
import shutil
import os
import json

def train_model(hyp, trial_num, use_default=False):
    trial_name = f"trial_{trial_num}"
    project_dir = f"runs/train/{trial_name}"

    # Hapus direktori sebelumnya jika ada
    if os.path.exists(project_dir):
        shutil.rmtree(project_dir)

    hyp_path = None
    if not use_default:
        hyp_path = f"{trial_name}_hyp.yaml"
        with open(hyp_path, 'w') as f:
            yaml.dump(hyp, f)

    # Gunakan device Apple MPS jika tersedia
    device = "mps" if torch.backends.mps.is_available() else "cpu"

    # Load model
    model = YOLO("yolo11n.pt")

    # Train
    results = model.train(
        data=dataset,
        epochs=100,
        imgsz=640,
        batch=16,
        name=trial_name,
        cfg=hyp_path if hyp_path else None,
        patience=10,  # Early stopping
        device=device,
    )

    # Simpan semua hasil
    try:
        result_dir = os.path.join("runs/train", trial_name)

        # Salin best.pt ke lokasi terpisah jika ingin
        best_weight = os.path.join(result_dir, "weights", "best.pt")
        if os.path.exists(best_weight):
            shutil.copy(best_weight, f"{trial_name}_best.pt")

        # Simpan metrics.json sebagai dict
        metrics_json = os.path.join(result_dir, "metrics.json")
        if os.path.exists(metrics_json):
            with open(metrics_json, 'r') as f:
                metrics_dict = json.load(f)
        else:
            metrics_dict = {}

        # Simpan confusion matrix
        cm_file = os.path.join(result_dir, "confusion_matrix.png")
        if os.path.exists(cm_file):
            shutil.copy(cm_file, f"{trial_name}_confusion_matrix.png")

        # Simpan CSV results
        csv_file = os.path.join(result_dir, "results.csv")
        if os.path.exists(csv_file):
            shutil.copy(csv_file, f"{trial_name}_results.csv")

        # Return mAP@50 jika tersedia
        return metrics_dict.get("metrics/mAP50(B)", 0.0)

    except Exception as e:
        print(f"⚠️ Error saving results for trial {trial_num}: {e}")
        return 0.0


def objective(trial):
    if trial.number == 0:
        print("🚀 Running baseline trial with default YOLOv11 hyperparameters...")
        return train_model(None, trial.number, use_default=True)

    hyp = {
        "lr0": trial.suggest_float('lr0', 1e-5, 1e-1, log=True),
        "momentum": trial.suggest_float('momentum', 0.80, 0.99),
        "weight_decay": trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
        "box": trial.suggest_float('box', 0.02, 0.4),
        "cls": trial.suggest_float('cls', 0.2, 1.0),
        "hsv_h": trial.suggest_float('hsv_h', 0.0, 0.1),
        "hsv_s": trial.suggest_float('hsv_s', 0.0, 0.7),
        "hsv_v": trial.suggest_float('hsv_v', 0.0, 0.4),
    }

    try:
        return train_model(hyp, trial.number)
    except Exception as e:
        print(f"⚠️ Trial {trial.number} failed: {e}")
        return 0.0

In [ ]:
import torch


device = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Using device: {device}")

In [ ]:
study = optuna.create_study(direction="maximize", study_name="YOLOv8_Tuning")
study.optimize(objective, n_trials=15)  # try 15 trials

In [ ]:
best_trial = study.best_trial
best_trial_number = best_trial.number
best_model_path = f"runs/detect/trial_{best_trial.number}/weights/best.pt"

In [ ]:
best_model = YOLO(best_model_path)
val_results = best_model.val(data=dataset, conf=0.25)

In [ ]:
print(f"Best trial number: {best_trial_number}")
print(f"Best model path: {best_model_path}")

## This Section is for Loop A Model for Validations

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # go one level up

from utils import normalizer, reset, validation 
from ultralytics import YOLO

In [ ]:
#replace with best(detect) or segment
selection = "segment"

In [ ]:
model = YOLO(f"/Users/user/Documents/GitHub/MARROWS/Models/{selection}/best.pt")

### Loop A Model for Multiple Data Variant Prediction

In [ ]:
subset_list = ["Sparse", "Normal", "Clumpped"]
# subset_list = ["CombineTest"]

for subset in subset_list:
    results = model.predict(
        source=f"/Users/user/Documents/GitHub/MARROWS/Data/{subset}/images/val",
        save=True,
        save_txt=True,
        save_conf=True,
        project=f"runs/detect/afif_{selection}",
        name=f"predict{subset}",
        exist_ok=True
    )

### Loop A Model for Multiple Validatio Data Variant Annotation vs Prediciton Count

In [ ]:

subset_list = ["Sparse", "Normal", "Clumpped","CombineTest"]

for subset in subset_list:
    validation.compare_annotation_vs_prediction(
        gt_label_folder=f"runs/detect/afif_{selection}/predict{subset}/actual_labels",  # Ground truth
        pred_label_folder=f"runs/detect/afif_{selection}/predict{subset}/labels",  # prediction
        output_img_path=f"{selection}_loss_ratio_plot_{subset}.png"
    )

### Loop A Model for Multiple Validation Data Variant

In [ ]:
# from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

model_path = "best.pt"
subset_paths = {
    "Sparse": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Sparse",
    "Normal": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Normal",
    "Clumpped": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Clumpped",
    "CombineTest": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/CombineTest"
}
yaml_paths = {k: os.path.join(v, "data.yaml") for k, v in subset_paths.items()}

summary_data = []
roc_data = []

for subset_name in subset_paths.keys():
    print(f"🔍 Evaluating: {subset_name}")
    
    # Run model validation
    results = model.val(data=yaml_paths[subset_name],save_json=True , split=f"val", save=False)

   

### Loop A Model for Multiple Data Variant ROC Analysis

In [ ]:
# subset_list = ["Sparse", "Normal", "Clumpped"]
# subset_list = ["CombineTest"]
subset_list = ["Sparse", "Normal", "Clumpped", "CombineTest"]

for subset in subset_list:
    print(f"\n🚀 Processing subset: {subset}")
    validation.run_roc_analysis(
        yaml_path=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/{subset}/data.yaml",
        gt_folder=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/{subset}/labels/val",
        pred_folder=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/notebooks/runs/detect/afif_{selection}/predict{subset}/labels",
        subset_name=subset,
        selection=selection,
    )